# 🧠 Train CIFAR-10 Image Classifier

Welcome! In this notebook, you'll learn how to:
1. Load and explore the CIFAR-10 dataset
2. Build a Convolutional Neural Network (CNN)
3. Train the model to recognize 10 different objects
4. Save the trained model for later use

## What is CIFAR-10?
CIFAR-10 is a dataset of 60,000 small images (32x32 pixels) across 10 categories:
- ✈️ Airplane
- 🚗 Automobile
- 🐦 Bird
- 🐱 Cat
- 🦌 Deer
- 🐕 Dog
- 🐸 Frog
- 🐴 Horse
- 🚢 Ship
- 🚚 Truck

## Step 1: Import Required Libraries

We need TensorFlow for deep learning, NumPy for numerical operations, and Matplotlib for visualization.

In [ ]:
# Deep Learning framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# Data processing
import numpy as np
import matplotlib.pyplot as plt

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Step 2: Load the CIFAR-10 Dataset

TensorFlow makes it easy to load CIFAR-10 - no need to download anything manually!

In [ ]:
# Load the dataset (this will download it automatically the first time)
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Print dataset shapes
print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

# Class names for CIFAR-10
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

## Step 3: Visualize Some Images

Let's see what our data looks like!

In [ ]:
# Display first 25 images from the training set
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_train[i])
    plt.xlabel(class_names[y_train[i][0]])
plt.tight_layout()
plt.show()

## Step 4: Normalize the Data

Neural networks work better when input values are small. We'll scale pixel values from 0-255 to 0-1.

In [ ]:
# Normalize pixel values to be between 0 and 1
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"Pixel value range after normalization: [{X_train.min()}, {X_train.max()}]")

## Step 5: Build the CNN Model

### What is a CNN?
A Convolutional Neural Network (CNN) is designed for image processing. It has:
- **Convolutional layers**: Extract features like edges, shapes
- **Pooling layers**: Reduce image size while keeping important info
- **Dense layers**: Make the final classification decision

In [ ]:
def create_model():
    """
    Creates a CNN model for CIFAR-10 classification.
    
    Architecture:
    - 3 Convolutional blocks (Conv2D + MaxPooling)
    - 2 Dense layers for classification
    - Dropout for regularization
    """
    model = models.Sequential([
        # First convolutional block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        
        # Second convolutional block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Third convolutional block
        layers.Conv2D(64, (3, 3), activation='relu'),
        
        # Flatten the 3D output to 1D
        layers.Flatten(),
        
        # Dense layers for classification
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),  # Prevent overfitting
        
        # Output layer (10 classes)
        layers.Dense(10, activation='softmax')
    ])
    
    return model

# Create the model
model = create_model()

# Display model architecture
model.summary()

## Step 6: Compile the Model

Before training, we need to configure:
- **Optimizer**: How the model learns (Adam is a popular choice)
- **Loss function**: What we're trying to minimize
- **Metrics**: How we measure performance

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

## Step 7: Train the Model

Now comes the exciting part - training! This might take 5-10 minutes.

**What's happening?**
- **Epoch**: One complete pass through all training data
- **Batch**: A small subset of data processed at once
- **Accuracy**: Percentage of correct predictions

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs=20,  # Number of times to go through the entire dataset
    batch_size=64,  # Number of images processed at once
    validation_data=(X_test, y_test),  # Evaluate on test data after each epoch
    verbose=1
)

print("\n✅ Training completed!")

## Step 8: Visualize Training Results

Let's plot how the model's performance improved over time.

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot accuracy
ax1.plot(history.history['accuracy'], label='Training Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy Over Time')
ax1.legend()
ax1.grid(True)

# Plot loss
ax2.plot(history.history['loss'], label='Training Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Model Loss Over Time')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Step 9: Evaluate the Model

Let's see how well our model performs on unseen test data.

In [ ]:
# Evaluate on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"\n📊 Test Results:")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

## Step 10: Make Predictions

Let's test our model on some random images from the test set!

In [ ]:
# Make predictions on test data
predictions = model.predict(X_test[:25])

# Visualize predictions
plt.figure(figsize=(12, 12))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_test[i])
    
    predicted_label = np.argmax(predictions[i])
    true_label = y_test[i][0]
    confidence = np.max(predictions[i]) * 100
    
    # Color code: green if correct, red if wrong
    color = 'green' if predicted_label == true_label else 'red'
    
    plt.xlabel(f"{class_names[predicted_label]}\n({confidence:.1f}%)", 
               color=color, fontsize=9)

plt.tight_layout()
plt.show()

## Step 11: Save the Model

Let's save our trained model so we can use it later for real-time webcam detection!

In [ ]:
# Save the model
model.save('../models/cifar10_model.h5')
print("✅ Model saved to '../models/cifar10_model.h5'")

# Also save in TensorFlow SavedModel format (more flexible)
model.save('../models/cifar10_model')
print("✅ Model also saved in SavedModel format to '../models/cifar10_model'")

## Step 12: Test Loading the Model

Let's verify we can load the saved model.

In [ ]:
# Load the saved model
loaded_model = keras.models.load_model('../models/cifar10_model.h5')

# Test it on a single image
test_image = X_test[0:1]
prediction = loaded_model.predict(test_image)
predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction) * 100

print(f"✅ Model loaded successfully!")
print(f"Test prediction: {predicted_class} ({confidence:.2f}% confidence)")

## 🎉 Congratulations!

You've successfully:
1. ✅ Loaded the CIFAR-10 dataset
2. ✅ Built a CNN from scratch
3. ✅ Trained the model
4. ✅ Evaluated its performance
5. ✅ Saved the model for later use

## 🚀 Next Steps

1. **Improve accuracy**: Try adding more layers, data augmentation, or training longer
2. **Use the model**: Check out the webcam app to see real-time predictions!
3. **Experiment**: Change hyperparameters and see how it affects performance

## 💡 Tips for Beginners

- **Low accuracy?** Try training for more epochs or adjusting the learning rate
- **Overfitting?** Add more dropout layers or use data augmentation
- **Want to learn more?** Check out TensorFlow tutorials at tensorflow.org

Happy learning! 🎓